In [1]:
import requests
import pandas as pd

resp = requests.get(
    "https://api-v3.mbta.com/alerts",
    params={"filter[activity]": "ALL"}
)
resp.raise_for_status()

rows = [
    {
        "id": a["id"],
        "effect": a["attributes"]["effect"],
        "severity": a["attributes"]["severity"],
        "header": a["attributes"]["header"],
    }
    for a in resp.json()["data"]
]
alerts = pd.DataFrame(rows)
print("Total active alerts:", len(alerts))

elevators = alerts[alerts["effect"] == "ELEVATOR_CLOSURE"]
print("Elevator outages right now:", len(elevators))
elevators.head(10)

Total active alerts: 159
Elevator outages right now: 12


,id,effect,severity,header
0,1031519,ELEVATOR_CLOSURE,3,"On Tuesday, September 22, through Tuesday, Dec..."
9,1036024,ELEVATOR_CLOSURE,3,Natick Center Elevator 751 (Track 2 to street)...
22,1033322,ELEVATOR_CLOSURE,8,South Acton Elevator 704 (Track 1 (outbound pl...
24,1030968,ELEVATOR_CLOSURE,3,Salem Elevator 997 (Bridge St to parking garag...
36,1036639,ELEVATOR_CLOSURE,3,Revere Beach Elevator 885 (Wonderland platform...
54,1035935,ELEVATOR_CLOSURE,3,JFK/UMass Elevator 830 (Braintree branch platf...
56,1002058,ELEVATOR_CLOSURE,3,Winchester Center Elevator 749 (Track 1 (outbo...
83,1031518,ELEVATOR_CLOSURE,3,"On Tuesday, September 22, through Tuesday, Dec..."
84,1037615,ELEVATOR_CLOSURE,3,Aquarium Elevator 913 (Wonderland platform to ...
131,1023448,ELEVATOR_CLOSURE,3,Worcester Elevator 782 (Parking lot to pedestr...


In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

resp = requests.get(
    "https://api-v3.mbta.com/alerts",
    params={"filter[activity]": "ALL"},
    headers={"x-api-key": os.getenv("MBTA_API_KEY")}
)
resp.raise_for_status()

rows = [
    {
        "id": a["id"],
        "effect": a["attributes"]["effect"],
        "severity": a["attributes"]["severity"],
        "header": a["attributes"]["header"],
    }
    for a in resp.json()["data"]
]
alerts = pd.DataFrame(rows)
print("Total active alerts:", len(alerts))

elevators = alerts[alerts["effect"] == "ELEVATOR_CLOSURE"]
print("Elevator outages right now:", len(elevators))
elevators.head(10)

Total active alerts: 159
Elevator outages right now: 12


,id,effect,severity,header
0,1031519,ELEVATOR_CLOSURE,3,"On Tuesday, September 22, through Tuesday, Dec..."
9,1036024,ELEVATOR_CLOSURE,3,Natick Center Elevator 751 (Track 2 to street)...
22,1033322,ELEVATOR_CLOSURE,8,South Acton Elevator 704 (Track 1 (outbound pl...
24,1030968,ELEVATOR_CLOSURE,3,Salem Elevator 997 (Bridge St to parking garag...
36,1036639,ELEVATOR_CLOSURE,3,Revere Beach Elevator 885 (Wonderland platform...
54,1035935,ELEVATOR_CLOSURE,3,JFK/UMass Elevator 830 (Braintree branch platf...
56,1002058,ELEVATOR_CLOSURE,3,Winchester Center Elevator 749 (Track 1 (outbo...
83,1031518,ELEVATOR_CLOSURE,3,"On Tuesday, September 22, through Tuesday, Dec..."
84,1037615,ELEVATOR_CLOSURE,3,Aquarium Elevator 913 (Wonderland platform to ...
131,1023448,ELEVATOR_CLOSURE,3,Worcester Elevator 782 (Parking lot to pedestr...


In [3]:
pd.set_option("display.max_colwidth", None)
elevators[["header"]]

,header
0,"On Tuesday, September 22, through Tuesday, December 1, State Elevator 803 (Forest Hills platform to Washington Street, Milk Street) will be unavailable due to maintenance."
9,Natick Center Elevator 751 (Track 2 to street) is unavailable.
22,South Acton Elevator 704 (Track 1 (outbound platform) to pedestrian bridge) is unavailable.
24,Salem Elevator 997 (Bridge St to parking garage to busway and platforms) is unavailable due to maintenance.
36,Revere Beach Elevator 885 (Wonderland platform to lobby) is unavailable due to maintenance.
54,JFK/UMass Elevator 830 (Braintree branch platform to lobby) is unavailable due to maintenance.
56,Winchester Center Elevator 749 (Track 1 (outbound platform) to Main St) unavailable
83,"On Tuesday, September 22, through Tuesday, December 1, Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) will be unavailable due to maintenance."
84,Aquarium Elevator 913 (Wonderland platform to State Street lobby) is unavailable due to maintenance.
131,Worcester Elevator 782 (Parking lot to pedestrian bridge) is unavailable. Please use elevator in main building at 2 Washington Sq.


In [4]:
import os, requests
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_anthropic import ChatAnthropic

@tool
def get_elevator_outages() -> str:
    """Get all MBTA elevators that are currently out of service."""
    resp = requests.get(
        "https://api-v3.mbta.com/alerts",
        params={"filter[activity]": "ALL"},
        headers={"x-api-key": os.getenv("MBTA_API_KEY")},
    )
    outages = [
        a["attributes"]["header"]
        for a in resp.json()["data"]
        if a["attributes"]["effect"] == "ELEVATOR_CLOSURE"
    ]
    return "\n".join(outages) or "No elevator outages right now."

checker = create_agent(
    model=ChatAnthropic(model="claude-sonnet-5-5"),
    tools=[get_elevator_outages],
    system_prompt=(
        "You are the Accessibility Checker for TransitWise Boston. "
        "Use your tools to check live elevator status, and explain "
        "clearly whether a wheelchair user can use a station."
    ),
)

result = checker.invoke({
    "messages": [{"role": "user",
                  "content": "I use a wheelchair. Can I use JFK/UMass station today?"}]
})
print(result["messages"][-1].content)

[{'signature': 'CAQSowoKEAgSGAI4AUIIdGhpbmtpbmcSDCBoi+BCfNI6pRwerRoMyKs98hg3gmH5Xc3LIjCjSl3ldjL+QJwxfdl8S3HUSrshsWRYxoF4vzoJ2+RmmV0erKnQXF4woe66wNNw+NEqwAmgxqFL2Lf5WpuJ8hxpbXMeCvttq+6E8F8JP2EC4tzxX7xpvARPJ2AnqAaZMmf4J+Xn/diIEG9F5qXf8y6dV5/Z6/YQHYF/+ICIYTjBaEKs82ivuvvgBo7/bE/W4rEJMWr2AQ6g4xlSZ8akq9Gxz7zSKf3x1kHWV0AUzZxn9DYnNvh9ATTG51xeZUJlT0M8gNK70QD4u6RZW2ho9us4rD5VGyy9pJNVmIpB1ZPk51AmVKdz3e5zPTKeDRJEAQ/R3KXTlU3gX6whtQuc+cbLfxb/ZSBUBlgSqc+qD26JLHne6pFQWoNj/N4Guf+QYkV9bnwbUk467VhNIielktFXYpxJSFNS0JbzelHxreo3w4z5XbX9q1DogP3pZyy1kWuXkJn/wRji+dIMPnl6C0z723ukvEhLUq/h3wwVtHagFXGgYDeVzzHT5FAc/u7Ml8p3ZWJwixS2kz/I6zr/A4AJx5pW3XSHpIXeFMhARiLfyMU4bjl/YJfXbxEhOdrkTxJIEEYZxkngvUkc00EH/AHtgrAnesUdQr9uhZKDsJf54jxM2fK7ewXvdMCwlzxEX7WqJ9DRhKyGhGCz+0IwP8COMIVVIPPsQUQ4ohsVKzPH2thsPur1u6I4MI+OQdpdEOHjFwchOnk9eNEQG+3s256Vcg8Ngh4+Dc/QOZl9PJpWLO2NKVcNe+Kt3lb8IP/Ru5VN8rCEGRbbopnL0+e0wOEb55F93SDgxhqAyI6kJkVa7JBo9ciUC7H3I/HYsDmFsHas+UgqSqEQatnDJ71DVjyFwlz8Zn1Y4DuCgOCcVI6ckrD4wDtigr6S5qn2JjkUtwsIdGN1eVhr9g6AyPJO

In [5]:
import os, requests
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_anthropic import ChatAnthropic

@tool
def get_elevator_outages() -> str:
    """Get all MBTA elevators that are currently out of service."""
    resp = requests.get(
        "https://api-v3.mbta.com/alerts",
        params={"filter[activity]": "ALL"},
        headers={"x-api-key": os.getenv("MBTA_API_KEY")},
    )
    outages = [
        a["attributes"]["header"]
        for a in resp.json()["data"]
        if a["attributes"]["effect"] == "ELEVATOR_CLOSURE"
    ]
    return "\n".join(outages) or "No elevator outages right now."

checker = create_agent(
    model=ChatAnthropic(model="claude-sonnet-5-5"),
    tools=[get_elevator_outages],
    system_prompt=(
        "You are the Accessibility Checker for TransitWise Boston. "
        "Use your tools to check live elevator status, and explain "
        "clearly whether a wheelchair user can use a station."
    ),
)

result = checker.invoke({
    "messages": [{"role": "user",
                  "content": "I use a wheelchair. Can I use JFK/UMass station today?"}]
})
print(result["messages"][-1].text)

**It depends on which branch you need.** JFK/UMass has one elevator out of service right now.

**Out of service:** Elevator 830, which links the **Braintree branch platform** to the lobby. It's down for maintenance, and the alert gives no return date.

**What that means for you:**
- **Braintree-branch platform:** You can't count on getting between the lobby and this platform today. If you're boarding or leaving a train there, you may not be able to get to or from the street.
- **Ashmont-branch platform:** No outage is reported for its elevator, so it should be usable. The outage list only covers elevators that are out, so I can't confirm the other elevators are working.

**Suggestions:**
- **Heading to Braintree or Quincy:** Think about boarding at a different station. You could also take an Ashmont-branch train and transfer at a station with working elevators. Check that the transfer station is accessible first, and I can do that for you.
- **Traveling to JFK/UMass on a Braintree trai

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
import snowflake.connector

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA"),
)

cur = conn.cursor()
cur.execute("SELECT CURRENT_USER(), CURRENT_DATABASE(), CURRENT_SCHEMA()")
print("Connected!", cur.fetchone())

Connected! ('TEJASTADLA', 'TRANSITWISE_DB', 'RAW')


In [8]:
from snowflake.connector.pandas_tools import write_pandas
from datetime import datetime, timezone

alerts["loaded_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
alerts.columns = [c.upper() for c in alerts.columns]

success, chunks, rows, _ = write_pandas(
    conn, alerts, "ALERTS_HISTORY",
    auto_create_table=True,
    use_logical_type=True,
)
print("Loaded:", success, "| Rows:", rows)

Loaded: True | Rows: 159


In [10]:
resp = requests.get(
    "https://api-v3.mbta.com/routes",
    params={"filter[type]": "0,1,2"},   # 0 = light rail, 1 = subway, 2 = commuter rail
    headers={"x-api-key": os.getenv("MBTA_API_KEY")},
)
resp.raise_for_status()

routes = pd.DataFrame([
    {
        "route_id": r["id"],
        "long_name": r["attributes"]["long_name"],
        "short_name": r["attributes"]["short_name"],
        "route_type": r["attributes"]["type"],
        "description": r["attributes"]["description"],
        "color": r["attributes"]["color"],
        "direction_names": ", ".join(r["attributes"]["direction_names"] or []),
    }
    for r in resp.json()["data"]
])

routes["loaded_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
routes.columns = [c.upper() for c in routes.columns]

print("Routes found:", len(routes))
display(routes.head(20))

success, chunks, rows, _ = write_pandas(
    conn, routes, "ROUTES",
    auto_create_table=True,
    overwrite=True,
    use_logical_type=True,
)
print("Loaded:", success, "| Rows:", rows)

Routes found: 21


,ROUTE_ID,LONG_NAME,SHORT_NAME,ROUTE_TYPE,DESCRIPTION,COLOR,DIRECTION_NAMES,LOADED_AT
0,Red,Red Line,,1,Rapid Transit,DA291C,"South, North",2026-10-08 15:52:12.870438
1,Mattapan,Mattapan Line,,0,Rapid Transit,DA291C,"Outbound, Inbound",2026-10-08 15:52:12.870438
2,Orange,Orange Line,,1,Rapid Transit,ED8B00,"South, North",2026-10-08 15:52:12.870438
3,Green-B,Green Line B,B,0,Rapid Transit,00843D,"West, East",2026-10-08 15:52:12.870438
4,Green-C,Green Line C,C,0,Rapid Transit,00843D,"West, East",2026-10-08 15:52:12.870438
5,Green-D,Green Line D,D,0,Rapid Transit,00843D,"West, East",2026-10-08 15:52:12.870438
6,Green-E,Green Line E,E,0,Rapid Transit,00843D,"West, East",2026-10-08 15:52:12.870438
7,Blue,Blue Line,,1,Rapid Transit,003DA5,"West, East",2026-10-08 15:52:12.870438
8,CR-Fairmount,Fairmount Line,,2,Regional Rail,80276C,"Outbound, Inbound",2026-10-08 15:52:12.870438
9,CR-NewBedford,Fall River/New Bedford Line,,2,Regional Rail,80276C,"Outbound, Inbound",2026-10-08 15:52:12.870438


Loaded: True | Rows: 21


In [19]:
resp = requests.get(
    "https://api-v3.mbta.com/stops",
    params={
        "filter[route_type]": "0,1,2",   # light rail, subway, commuter rail
        "include": "parent_station",
    },
    headers={"x-api-key": os.getenv("MBTA_API_KEY")},
)
resp.raise_for_status()
data = resp.json()
parents = {p["id"]: p for p in data.get("included", []) if p["type"] == "stop"}
for s in data["data"]:
    if not s["relationships"]["parent_station"]["data"]:
        parents.setdefault(s["id"], s)   # stops that are already a station
station_list = list(parents.values())

access_labels = {0: "unknown", 1: "accessible", 2: "not accessible"}

stations = pd.DataFrame([
    {
        "station_id": s["id"],
        "name": s["attributes"]["name"],
        "municipality": s["attributes"]["municipality"],
        "address": s["attributes"]["address"],
        "latitude": s["attributes"]["latitude"],
        "longitude": s["attributes"]["longitude"],
        "wheelchair_boarding": s["attributes"]["wheelchair_boarding"],
        "wheelchair_status": access_labels.get(s["attributes"]["wheelchair_boarding"], "unknown"),
    }
     for s in station_list
])

stations["loaded_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
stations.columns = [c.upper() for c in stations.columns]

print("Stations found:", len(stations))
print(stations["WHEELCHAIR_STATUS"].value_counts())
display(stations)

success, chunks, rows, _ = write_pandas(
    conn, stations, "STATIONS",
    auto_create_table=True,
    overwrite=True,
    use_logical_type=True,
)
print("Loaded:", success, "| Rows:", rows)

Stations found: 257
WHEELCHAIR_STATUS
accessible        211
not accessible     46
Name: count, dtype: int64


,STATION_ID,NAME,MUNICIPALITY,ADDRESS,LATITUDE,LONGITUDE,WHEELCHAIR_BOARDING,WHEELCHAIR_STATUS,LOADED_AT
0,place-sstat,South Station,Boston,"700 Atlantic Ave, Boston, MA 02110",42.352271,-71.055242,1,accessible,2026-10-08 16:06:49.434171
1,place-WML-0199,West Natick,Natick,"249 West Central St, Natick, MA 01760",42.283064,-71.391797,1,accessible,2026-10-08 16:06:49.434171
2,place-cedgr,Cedar Grove,Boston,"Fellsway St and Milton St, Dorchester, MA 02124",42.279629,-71.060394,1,accessible,2026-10-08 16:06:49.434171
3,place-FB-0143,Norwood Depot,Norwood,"14 Hill St, Norwood, MA 02062",42.196857,-71.196688,1,accessible,2026-10-08 16:06:49.434171
4,place-chncl,Chinatown,Boston,"Washington St and Boylston St, Boston, MA",42.352547,-71.062752,1,accessible,2026-10-08 16:06:49.434171
...,...,...,...,...,...,...,...,...,...
252,place-NBM-0374,East Taunton,Taunton,"1137 County Street, Taunton, MA 02780",41.868197,-71.061694,1,accessible,2026-10-08 16:06:49.434171
253,place-WR-0067,Melrose/Cedar Park,Melrose,"Cedar Park and West Emerson St, Melrose, MA 02176",42.458768,-71.069789,2,not accessible,2026-10-08 16:06:49.434171
254,place-brdwy,Broadway,Boston,"Dorchester Ave and Broadway, Boston, MA",42.342622,-71.056967,1,accessible,2026-10-08 16:06:49.434171
255,place-hsmnl,Heath Street,Boston,"Heath St and South Huntington Ave, Boston, MA",42.328316,-71.110252,1,accessible,2026-10-08 16:06:49.434171


Loaded: True | Rows: 257


In [20]:
stations[stations["NAME"].str.contains("Park Street")][["STATION_ID", "NAME"]]

,STATION_ID,NAME
129,place-pktrm,Park Street


In [23]:
rows = []
for route_id in routes["ROUTE_ID"]:
    r = requests.get(
        "https://api-v3.mbta.com/stops",
        params={"filter[route]": route_id, "filter[direction_id]": "0"},
        headers={"x-api-key": os.getenv("MBTA_API_KEY")},
    )
    r.raise_for_status()
    for seq, s in enumerate(r.json()["data"], start=1):
        rows.append({
            "route_id": route_id,
            "stop_sequence": seq,
            "station_id": s["id"],
            "station_name": s["attributes"]["name"],
        })

route_stops = pd.DataFrame(rows)
route_stops["loaded_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
route_stops.columns = [c.upper() for c in route_stops.columns]

print("Route-station links:", len(route_stops))
display(route_stops[route_stops["ROUTE_ID"] == "Red"])

missing = set(route_stops["STATION_ID"]) - set(stations["STATION_ID"])
print("Stations not found in STATIONS table:", len(missing))

success, chunks, rows_loaded, _ = write_pandas(
    conn, route_stops, "ROUTE_STOPS",
    auto_create_table=True,
    overwrite=True,
    use_logical_type=True,
)
print("Loaded:", success, "| Rows:", rows_loaded)

Route-station links: 337


,ROUTE_ID,STOP_SEQUENCE,STATION_ID,STATION_NAME,LOADED_AT
0,Red,1,place-alfcl,Alewife,2026-10-08 16:14:21.626234
1,Red,2,place-davis,Davis,2026-10-08 16:14:21.626234
2,Red,3,place-portr,Porter,2026-10-08 16:14:21.626234
3,Red,4,place-harsq,Harvard,2026-10-08 16:14:21.626234
4,Red,5,place-cntsq,Central,2026-10-08 16:14:21.626234
5,Red,6,place-knncl,Kendall/MIT,2026-10-08 16:14:21.626234
6,Red,7,place-chmnl,Charles/MGH,2026-10-08 16:14:21.626234
7,Red,8,place-pktrm,Park Street,2026-10-08 16:14:21.626234
8,Red,9,place-dwnxg,Downtown Crossing,2026-10-08 16:14:21.626234
9,Red,10,place-sstat,South Station,2026-10-08 16:14:21.626234


Stations not found in STATIONS table: 0
Loaded: True | Rows: 337


In [24]:
rows = []
for route_id in routes["ROUTE_ID"]:
    r = requests.get(
        "https://api-v3.mbta.com/stops",
        params={"filter[route]": route_id, "filter[direction_id]": "0"},
        headers={"x-api-key": os.getenv("MBTA_API_KEY")},
    )
    r.raise_for_status()
    for seq, s in enumerate(r.json()["data"], start=1):
        rows.append({
            "route_id": route_id,
            "stop_sequence": seq,
            "station_id": s["id"],
            "station_name": s["attributes"]["name"],
        })

route_stops = pd.DataFrame(rows)
route_stops["loaded_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
route_stops.columns = [c.upper() for c in route_stops.columns]

print("Route-station links:", len(route_stops))
display(route_stops[route_stops["ROUTE_ID"] == "Orange"])

missing = set(route_stops["STATION_ID"]) - set(stations["STATION_ID"])
print("Stations not found in STATIONS table:", len(missing))

success, chunks, rows_loaded, _ = write_pandas(
    conn, route_stops, "ROUTE_STOPS",
    auto_create_table=True,
    overwrite=True,
    use_logical_type=True,
)
print("Loaded:", success, "| Rows:", rows_loaded)

Route-station links: 337


,ROUTE_ID,STOP_SEQUENCE,STATION_ID,STATION_NAME,LOADED_AT
30,Orange,1,place-ogmnl,Oak Grove,2026-10-08 16:15:18.894766
31,Orange,2,place-mlmnl,Malden Center,2026-10-08 16:15:18.894766
32,Orange,3,place-welln,Wellington,2026-10-08 16:15:18.894766
33,Orange,4,place-astao,Assembly,2026-10-08 16:15:18.894766
34,Orange,5,place-sull,Sullivan Square,2026-10-08 16:15:18.894766
35,Orange,6,place-ccmnl,Community College,2026-10-08 16:15:18.894766
36,Orange,7,place-north,North Station,2026-10-08 16:15:18.894766
37,Orange,8,place-haecl,Haymarket,2026-10-08 16:15:18.894766
38,Orange,9,place-state,State,2026-10-08 16:15:18.894766
39,Orange,10,place-dwnxg,Downtown Crossing,2026-10-08 16:15:18.894766


Stations not found in STATIONS table: 0
Loaded: True | Rows: 337


In [27]:
rows = []
for route_id in routes["ROUTE_ID"]:
    r = requests.get(
        "https://api-v3.mbta.com/stops",
        params={"filter[route]": route_id, "filter[direction_id]": "0"},
        headers={"x-api-key": os.getenv("MBTA_API_KEY")},
    )
    r.raise_for_status()
    for seq, s in enumerate(r.json()["data"], start=1):
        rows.append({
            "route_id": route_id,
            "stop_sequence": seq,
            "station_id": s["id"],
            "station_name": s["attributes"]["name"],
        })

route_stops = pd.DataFrame(rows)
route_stops["loaded_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
route_stops.columns = [c.upper() for c in route_stops.columns]

print("Route-station links:", len(route_stops))
display(route_stops[route_stops["ROUTE_ID"].isin(
    ["Green-E"]
)])

missing = set(route_stops["STATION_ID"]) - set(stations["STATION_ID"])
print("Stations not found in STATIONS table:", len(missing))

success, chunks, rows_loaded, _ = write_pandas(
    conn, route_stops, "ROUTE_STOPS",
    auto_create_table=True,
    overwrite=True,
    use_logical_type=True,
)
print("Loaded:", success, "| Rows:", rows_loaded)

Route-station links: 337


,ROUTE_ID,STOP_SEQUENCE,STATION_ID,STATION_NAME,LOADED_AT
118,Green-E,1,place-mdftf,Medford/Tufts,2026-10-08 16:16:57.727707
119,Green-E,2,place-balsq,Ball Square,2026-10-08 16:16:57.727707
120,Green-E,3,place-mgngl,Magoun Square,2026-10-08 16:16:57.727707
121,Green-E,4,place-gilmn,Gilman Square,2026-10-08 16:16:57.727707
122,Green-E,5,place-esomr,East Somerville,2026-10-08 16:16:57.727707
123,Green-E,6,place-lech,Lechmere,2026-10-08 16:16:57.727707
124,Green-E,7,place-spmnl,Science Park/West End,2026-10-08 16:16:57.727707
125,Green-E,8,place-north,North Station,2026-10-08 16:16:57.727707
126,Green-E,9,place-haecl,Haymarket,2026-10-08 16:16:57.727707
127,Green-E,10,place-gover,Government Center,2026-10-08 16:16:57.727707


Stations not found in STATIONS table: 0
Loaded: True | Rows: 337


In [28]:
route_stops["ROUTE_ID"].unique()

array(['Red', 'Mattapan', 'Orange', 'Green-B', 'Green-C', 'Green-D',
       'Green-E', 'Blue', 'CR-Fairmount', 'CR-NewBedford', 'CR-Fitchburg',
       'CR-Worcester', 'CR-Franklin', 'CR-Greenbush', 'CR-Haverhill',
       'CR-Kingston', 'CR-Lowell', 'CR-Needham', 'CR-Newburyport',
       'CR-Providence', 'CR-Foxboro'], dtype=object)

In [31]:
resp = requests.get(
    "https://api-v3.mbta.com/facilities",
    params={"filter[type]": "ELEVATOR,ESCALATOR,PORTABLE_BOARDING_LIFT,ELEVATED_SUBPLATFORM"},
    headers={"x-api-key": os.getenv("MBTA_API_KEY")},
)
resp.raise_for_status()

facilities = pd.DataFrame([
    {
        "facility_id": f["id"],
        "type": f["attributes"]["type"],
        "short_name": f["attributes"]["short_name"],
        "long_name": f["attributes"]["long_name"],
        "station_id": (f["relationships"]["stop"]["data"] or {}).get("id"),
    }
    for f in resp.json()["data"]
])

facilities["loaded_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
facilities.columns = [c.upper() for c in facilities.columns]

# keep only facilities at stations in your STATIONS table
facilities = facilities[facilities["STATION_ID"].isin(stations["STATION_ID"])].reset_index(drop=True)

print("Facilities found:", len(facilities))
print(facilities["TYPE"].value_counts())
display(facilities.head(20))

success, chunks, rows_loaded, _ = write_pandas(
    conn, facilities, "FACILITIES",
    auto_create_table=True,
    overwrite=True,
    use_logical_type=True,
)
print("Loaded:", success, "| Rows:", rows_loaded)

Facilities found: 584
TYPE
ELEVATOR                  228
ESCALATOR                 172
ELEVATED_SUBPLATFORM      128
PORTABLE_BOARDING_LIFT     56
Name: count, dtype: int64


,FACILITY_ID,TYPE,SHORT_NAME,LONG_NAME,STATION_ID,LOADED_AT
0,947,ELEVATOR,Sydney Street to unpaid lobby,Savin Hill Elevator 947 (Sydney Street to unpaid lobby),place-shmnl,2026-10-08 17:08:32.485283
1,portlift-brmnl-1,PORTABLE_BOARDING_LIFT,Mobile lift (Park Street & North),Brigham Circle mobile lift (Park Street & North),place-brmnl,2026-10-08 17:08:32.485283
2,821,ELEVATOR,Lobby to Harvard Square,Harvard Elevator 821 (Lobby to Harvard Square),place-harsq,2026-10-08 17:08:32.485283
3,subplat-GB-0296-1,ELEVATED_SUBPLATFORM,Mini-high platform (Track 1/Outbound),West Gloucester mini-high platform (Track 1/Outbound),place-GB-0296,2026-10-08 17:08:32.485283
4,subplat-NHRML-0218-2,ELEVATED_SUBPLATFORM,Mini-high platform,North Billerica mini-high platform,place-NHRML-0218,2026-10-08 17:08:32.485283
5,147,ESCALATOR,Forest Hills platform to Washington Street lobby,Tufts Medical Center Escalator 147 (Forest Hills platform to Washington Street lobby),place-tumnl,2026-10-08 17:08:32.485283
6,765,ELEVATOR,Platform to Medford Street lobby,Gilman Square Elevator 765 (Platform to Medford Street lobby),place-gilmn,2026-10-08 17:08:32.485283
7,510,ESCALATOR,Ashmont/Braintree platform to paid lobby,Porter Escalator 510 (Ashmont/Braintree platform to paid lobby),place-portr,2026-10-08 17:08:32.485283
8,997,ELEVATOR,Bridge St to parking garage to busway and platforms,Salem Elevator 997 (Bridge St to parking garage to busway and platforms),place-ER-0168,2026-10-08 17:08:32.485283
9,516,ESCALATOR,Alewife platform to Church Street lobby,Harvard Escalator 516 (Alewife platform to Church Street lobby),place-harsq,2026-10-08 17:08:32.485283


Loaded: True | Rows: 584


In [32]:
print("Stations in STATIONS:", stations["STATION_ID"].nunique())
print("Stations on routes:", route_stops["STATION_ID"].nunique())

Stations in STATIONS: 257
Stations on routes: 257


In [34]:
esp = requests.get(
    "https://api-v3.mbta.com/alerts",
    params={"filter[activity]": "ALL"},
    headers={"x-api-key": os.getenv("MBTA_API_KEY")},
)
resp.raise_for_status()

def ids(entities, key):
    return ",".join(sorted({e[key] for e in entities if e.get(key)}))

rows = []
for a in resp.json()["data"]:
    attr = a["attributes"]
    ents = attr.get("informed_entity") or []
    periods = attr.get("active_period") or []
    rows.append({
        "id": a["id"],
        "effect": attr["effect"],
        "cause": attr.get("cause"),
        "severity": attr["severity"],
        "lifecycle": attr.get("lifecycle"),
        "header": attr["header"],
        "description": attr.get("description"),
        "facility_ids": ids(ents, "facility"),
        "station_ids": ids(ents, "stop"),
        "route_ids": ids(ents, "route"),
        "active_start": periods[0]["start"] if periods else None,
        "active_end": periods[-1]["end"] if periods else None,
        "updated_at": attr.get("updated_at"),
    })
alerts = pd.DataFrame(rows)

KeyError: 'effect'

In [35]:
print(resp.url)
print(resp.json()["data"][0]["type"])
print(list(resp.json()["data"][0]["attributes"].keys()))

https://api-v3.mbta.com/facilities?filter%5Btype%5D=ELEVATOR%2CESCALATOR%2CPORTABLE_BOARDING_LIFT%2CELEVATED_SUBPLATFORM
facility
['latitude', 'long_name', 'longitude', 'properties', 'short_name', 'type']


In [36]:
import os
from datetime import datetime, timezone

import pandas as pd
import requests
import snowflake.connector
from dotenv import load_dotenv
from snowflake.connector.pandas_tools import write_pandas

load_dotenv(override=True)

# 1. Extract: get all current MBTA alerts
alerts_resp = requests.get(
    "https://api-v3.mbta.com/alerts",
    params={"filter[activity]": "ALL"},
    headers={"x-api-key": os.getenv("MBTA_API_KEY")},
    timeout=30,
)
alerts_resp.raise_for_status()

# 2. Transform: flatten each alert into one row
def ids(entities, key):
    return ",".join(sorted({e[key] for e in entities if e.get(key)}))

rows = []
for a in alerts_resp.json()["data"]:
    attr = a["attributes"]
    ents = attr.get("informed_entity") or []
    periods = attr.get("active_period") or []
    rows.append({
        "id": a["id"],
        "effect": attr.get("effect"),
        "cause": attr.get("cause"),
        "severity": attr.get("severity"),
        "lifecycle": attr.get("lifecycle"),
        "header": attr.get("header"),
        "description": attr.get("description"),
        "facility_ids": ids(ents, "facility"),
        "station_ids": ids(ents, "stop"),
        "route_ids": ids(ents, "route"),
        "active_start": periods[0]["start"] if periods else None,
        "active_end": periods[-1]["end"] if periods else None,
        "updated_at": attr.get("updated_at"),
    })

alerts = pd.DataFrame(rows)
alerts["loaded_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
alerts.columns = [c.upper() for c in alerts.columns]

print("Total alerts:", len(alerts))
display(alerts[alerts["EFFECT"] == "ELEVATOR_CLOSURE"][["HEADER", "FACILITY_IDS", "STATION_IDS"]])

# 3. Load: append this snapshot to Snowflake
conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA"),
)

success, chunks, rows_loaded, _ = write_pandas(
    conn, alerts, "ALERTS_HISTORY",
    auto_create_table=True,
    use_logical_type=True,
)
print("Loaded:", success, "| Rows:", rows_loaded)

Total alerts: 197


,HEADER,FACILITY_IDS,STATION_IDS
5,"On Tuesday, September 22, through Tuesday, December 1, State Elevator 803 (Forest Hills platform to Washington Street, Milk Street) will be unavailable due to maintenance.",803,"70022,70023,70041,70042,door-state-meeting,place-state"
14,Natick Center Elevator 751 (Track 2 to street) is unavailable.,751,"WML-0177-02,place-WML-0177"
31,South Acton Elevator 704 (Track 1 (outbound platform) to pedestrian bridge) is unavailable.,704,"FR-0253-01,FR-0253-B,place-FR-0253"
32,Salem Elevator 997 (Bridge St to parking garage to busway and platforms) is unavailable due to maintenance.,997,"37150,ER-0168-B,ER-0168-S,place-ER-0168"
45,Revere Beach Elevator 885 (Wonderland platform to lobby) is unavailable due to maintenance.,885,"70058,door-rbmnl-main,place-rbmnl"
66,JFK/UMass Elevator 830 (Braintree branch platform to lobby) is unavailable due to maintenance.,830,"70095,70096,place-jfk"
106,"On Tuesday, September 22, through Tuesday, December 1, Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) will be unavailable due to maintenance.",869,"70078,door-dwnxg-101arch,door-dwnxg-macys,door-dwnxg-rochebros,door-dwnxg-summer,door-dwnxg-summereast,place-dwnxg"
117,Quincy Center Elevator 810 (Red Line platform to lobby) is unavailable due to maintenance.,810,"70101,70102,place-qnctr"
160,Worcester Elevator 782 (Parking lot to pedestrian bridge) is unavailable. Please use elevator in main building at 2 Washington Sq.,782,"WML-0442-CS,place-WML-0442"


Loaded: True | Rows: 197
